Este archivo es para convertir el dataset de la dgis en informacion util para poder ser analizada en el EDA

In [1]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt

In [2]:
dgis = pd.read_csv("dgis_integrado.csv")
dgis

,ent_resid,clave_municipio,causa_def,sexo,edad,edad_agru,dia_ocurr,mes_ocurr,año,escolarida
0,9,7,E117,2,68,18,21,4,2015,1
1,9,4,E119,2,72,19,12,11,2015,5
2,9,5,E149,2,88,22,10,7,2015,3
3,9,5,E116,1,87,22,22,7,2015,5
4,9,10,E112,1,57,16,5,7,2015,3
...,...,...,...,...,...,...,...,...,...,...
61142,9,12,E117,1,50,15,16,11,2020,4
61143,9,17,E147,2,73,19,12,11,2020,6
61144,9,17,E116,1,81,21,16,12,2020,3
61145,9,2,E142,1,60,17,20,8,2020,8


Vamos a crear otro dataframe que contenga la info sintetizada del primero

In [3]:
dgis2= dgis

dgis2['clase_enf'] = dgis2['causa_def'].str.extract(r'(\d+)').astype(int)

def clasificar_diabetes(codigo):
    if 100 <= codigo <= 109:
        return 1
    elif 110 <= codigo <= 119:
        return 2
    elif 120 <= codigo <= 149:
        return 3 #'otras diabetes'
    else:
        return 'Fuera de rango'

dgis2['dmt'] = dgis2['clase_enf'].apply(clasificar_diabetes)

dgis2[['causa_def','clase_enf','dmt']].head(15)

,causa_def,clase_enf,dmt
0,E117,117,2
1,E119,119,2
2,E149,149,3
3,E116,116,2
4,E112,112,2
5,E149,149,3
6,E147,147,3
7,E117,117,2
8,E147,147,3
9,E119,119,2


In [4]:
#dgis2
dgis2 = dgis2[((dgis2["escolarida"] != 99) & (dgis2["dmt"] <= 2))]
dgis2

,ent_resid,clave_municipio,causa_def,sexo,edad,edad_agru,dia_ocurr,mes_ocurr,año,escolarida,clase_enf,dmt
0,9,7,E117,2,68,18,21,4,2015,1,117,2
1,9,4,E119,2,72,19,12,11,2015,5,119,2
3,9,5,E116,1,87,22,22,7,2015,5,116,2
4,9,10,E112,1,57,16,5,7,2015,3,112,2
7,9,2,E117,2,68,18,19,11,2015,9,117,2
...,...,...,...,...,...,...,...,...,...,...,...,...
61140,9,8,E112,1,52,15,10,8,2020,1,112,2
61141,9,7,E117,2,26,10,28,12,2020,6,117,2
61142,9,12,E117,1,50,15,16,11,2020,4,117,2
61144,9,17,E116,1,81,21,16,12,2020,3,116,2


In [5]:

# Suponiendo que tu dataframe original se llama 'df_dgis'

# 1. Crear categorías de edad
bins = [0, 14, 24, 59, 150]
labels = ['0-14', '15-24', '25-59', '60+']
dgis2['rango_edad'] = pd.cut(dgis2['edad'], bins=bins, labels=labels, right=True)

# 2. Asegurar que el sexo esté claro (ej. 1: Masculino, 2: Femenino)
# Ajusta estos valores según los códigos que traiga tu dataset de DGIS
#dgis2['sexo_txt'] = dgis2['sexo'].map({1: 'Masc', 2: 'Fem'})

C:\Users\super\AppData\Local\Temp\ipykernel_17240\2898514469.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dgis2['rango_edad'] = pd.cut(dgis2['edad'], bins=bins, labels=labels, right=True)


In [6]:
# Creamos variables auxiliares para facilitar el conteo
# Agrupamos por Municipio y Año
#resumen= dgis2.groupby(['clave_municipio', 'año']).agg(
    # Casos Totales por Tipo
#    dmt1_total=('dmt', lambda x: (x == 1).sum()),
#    dmt2_total=('dmt', lambda x: (x == 2).sum()),
#    otro_total=('dmt', lambda x: ((x != 1) & (x != 2)).sum()),
    
    # Casos Femeninos por Tipo
#    dmt1_fem=('dmt', lambda x: ((dgis2.loc[x.index, 'sexo'] == 2) & (x == 1)).sum()),
#    dmt2_fem=('dmt', lambda x: ((dgis2.loc[x.index, 'sexo'] == 2) & (x == 2)).sum()),
    
    
    # Casos Masculinos por Tipo
#    dmt1_masc=('dmt', lambda x: ((dgis2.loc[x.index, 'sexo'] == 1) & (x == 1)).sum()),
#    dmt2_masc=('dmt', lambda x: ((dgis2.loc[x.index, 'sexo'] == 1) & (x == 2)).sum()),
    
#).reset_index()

# Ahora añadimos los rangos de edad cruzados por sexo
# Esto se hace más fácil con un pivot_table y luego un join
#edad_sexo = pd.crosstab(
#    [dgis2['clave_municipio'], dgis2['año']], 
#    [dgis2['rango_edad'], dgis2['sexo']]
#)

# Aplanamos los nombres de las columnas (ej: ('0-14', 'Fem') -> '0-14_Fem')
#edad_sexo.columns = [f'{edad}_sexo{sexo}' for edad, sexo in edad_sexo.columns]
#edad_sexo = edad_sexo.reset_index()

# Unimos todo en el dataset final
#df_final = pd.merge(resumen, edad_sexo, on=['clave_municipio', 'año'])


resumen = dgis2.groupby(['clave_municipio', 'año']).agg(
    dmt1_total=('dmt', lambda x: (x == 1).sum()),
    dmt2_total=('dmt', lambda x: (x == 2).sum()),
    otro_total=('dmt', lambda x: ((x != 1) & (x != 2)).sum()),
    dmt1_fem=('dmt', lambda x: ((dgis2.loc[x.index, 'sexo'] == 2) & (x == 1)).sum()),
    dmt2_fem=('dmt', lambda x: ((dgis2.loc[x.index, 'sexo'] == 2) & (x == 2)).sum()),
    dmt1_masc=('dmt', lambda x: ((dgis2.loc[x.index, 'sexo'] == 1) & (x == 1)).sum()),
    dmt2_masc=('dmt', lambda x: ((dgis2.loc[x.index, 'sexo'] == 1) & (x == 2)).sum()),
).reset_index()


# Filtramos para que solo tome DMT 1 y 2 
dgis_filtrado = dgis2[dgis2['dmt'].isin([1, 2])]

edad_sexo_dmt = pd.crosstab(
    index=[dgis_filtrado['clave_municipio'], dgis_filtrado['año']], 
    columns=[dgis_filtrado['rango_edad'], dgis_filtrado['sexo'], dgis_filtrado['dmt']]
)

# 3. Ajustamos el generador de nombres de columnas
# Estructura: '0-14_sexo1_dmt1', '60ymas_sexo2_dmt2', etc.
edad_sexo_dmt.columns = [f'{edad}_sexo{sexo}_dmt{dmt}' for edad, sexo, dmt in edad_sexo_dmt.columns]
edad_sexo_dmt = edad_sexo_dmt.reset_index()

# 4. Unimos todo
df_final = pd.merge(resumen, edad_sexo_dmt, on=['clave_municipio', 'año'], how='left').fillna(0)

In [7]:
df_final["casos totales"] = df_final["dmt1_total"] +df_final["dmt2_total"] + df_final["otro_total"]
df_final.columns

Index(['clave_municipio', 'año', 'dmt1_total', 'dmt2_total', 'otro_total',
       'dmt1_fem', 'dmt2_fem', 'dmt1_masc', 'dmt2_masc', '0-14_sexo1_dmt1',
       '0-14_sexo1_dmt2', '0-14_sexo2_dmt1', '0-14_sexo2_dmt2',
       '15-24_sexo1_dmt1', '15-24_sexo1_dmt2', '15-24_sexo2_dmt1',
       '15-24_sexo2_dmt2', '25-59_sexo1_dmt1', '25-59_sexo1_dmt2',
       '25-59_sexo2_dmt1', '25-59_sexo2_dmt2', '60+_sexo1_dmt1',
       '60+_sexo1_dmt2', '60+_sexo2_dmt1', '60+_sexo2_dmt2', 'casos totales'],
      dtype='object')

In [8]:
#df_final = df_final["año"] <= 2015
df_final = df_final[(df_final["año"] >= 2015)]
df_final
#df_final

,clave_municipio,año,dmt1_total,dmt2_total,otro_total,dmt1_fem,dmt2_fem,dmt1_masc,dmt2_masc,0-14_sexo1_dmt1,...,15-24_sexo2_dmt2,25-59_sexo1_dmt1,25-59_sexo1_dmt2,25-59_sexo2_dmt1,25-59_sexo2_dmt2,60+_sexo1_dmt1,60+_sexo1_dmt2,60+_sexo2_dmt1,60+_sexo2_dmt2,casos totales
0,2,2015,8,424,0,5,191,3,233,0,...,0,2,63,3,27,1,170,2,164,432
1,2,2016,7,512,0,2,250,5,262,0,...,0,3,55,2,42,2,207,0,208,519
2,2,2017,8,453,0,3,211,5,242,0,...,1,3,53,0,26,2,189,3,184,461
3,2,2018,6,397,0,1,188,5,209,0,...,0,3,53,1,16,2,156,0,172,403
4,2,2019,7,364,0,3,176,4,188,0,...,0,2,58,2,31,2,130,1,145,371
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,17,2016,4,511,0,2,257,2,254,0,...,0,2,73,1,43,0,180,0,214,515
92,17,2017,12,449,0,5,221,7,228,0,...,0,2,59,3,29,4,169,2,192,461
93,17,2018,6,373,0,2,178,4,195,0,...,0,1,62,0,29,3,133,2,149,379
94,17,2019,7,477,0,5,235,2,242,0,...,0,0,79,1,37,1,163,2,198,484


In [9]:
v1= df_final[df_final['año'] == 2020]
v2 = v1['dmt2_total'].sum()
print(v2)

12837


In [10]:
# Definimos quién es Hombre (1) y Mujer (2)
es_masc = dgis2['sexo'] == 1
es_fem = dgis2['sexo'] == 2

# Definimos los tipos de DMT
es_dmt1 = dgis2['dmt'] == 1
es_dmt2 = dgis2['dmt'] == 2

# Creamos columnas auxiliares para los casos que pides
# Ejemplo: Hombre, DMT 1, Sin escolaridad (valor 1)
dgis2['m_dmt1_sin_esc'] = (es_masc & es_dmt1 & (dgis2['escolarida'] == 1)).astype(int)
dgis2['m_dmt1_preescolar'] = (es_masc & es_dmt1 & (dgis2['escolarida'] == 2)).astype(int)
dgis2['m_dmt1_primaria'] = (es_masc & es_dmt1 & (dgis2['escolarida'] > 2) & (dgis2['escolarida'] <= 4)).astype(int)
dgis2['m_dmt1_secundaria'] = (es_masc & es_dmt1 & (dgis2['escolarida'] > 4) & (dgis2['escolarida'] <= 6)).astype(int)
dgis2['m_dmt1_preparatoria'] = (es_masc & es_dmt1 & (dgis2['escolarida'] > 6) & (dgis2['escolarida'] <= 8)).astype(int)
dgis2['m_dmt1_profesional'] = (es_masc & es_dmt1 & (dgis2['escolarida'] == 9)).astype(int)
dgis2['m_dmt1_posgrado'] = (es_masc & es_dmt1 & (dgis2['escolarida'] == 10)).astype(int)

dgis2['m_dmt2_sin_esc'] = (es_masc & es_dmt2 & (dgis2['escolarida'] == 1)).astype(int)
dgis2['m_dmt2_preescolar'] = (es_masc & es_dmt2 & (dgis2['escolarida'] == 2)).astype(int)
dgis2['m_dmt2_primaria'] = (es_masc & es_dmt2 & (dgis2['escolarida'] > 2) & (dgis2['escolarida'] <= 4)).astype(int)
dgis2['m_dmt2_secundaria'] = (es_masc & es_dmt2 & (dgis2['escolarida'] > 4) & (dgis2['escolarida'] <= 6)).astype(int)
dgis2['m_dmt2_preparatoria'] = (es_masc & es_dmt2 & (dgis2['escolarida'] > 6) & (dgis2['escolarida'] <= 8)).astype(int)
dgis2['m_dmt2_profesional'] = (es_masc & es_dmt2 & (dgis2['escolarida'] == 9)).astype(int)
dgis2['m_dmt2_posgrado'] = (es_masc & es_dmt2 & (dgis2['escolarida'] == 10)).astype(int)

# Ejemplo: Mujer, DMT 1, Licenciatura (valor 9)
#dgis2['f_d1_lic'] = (es_fem & es_dmt1 & (dgis2['escolarida'] == 9)).astype(int)
dgis2['f_dmt1_sin_esc'] = (es_fem & es_dmt1 & (dgis2['escolarida'] == 1)).astype(int)
dgis2['f_dmt1_preescolar'] = (es_fem & es_dmt1 & (dgis2['escolarida'] == 2)).astype(int)
dgis2['f_dmt1_primaria'] = (es_fem & es_dmt1 & (dgis2['escolarida'] > 2) & (dgis2['escolarida'] <= 4)).astype(int)
dgis2['f_dmt1_secundaria'] = (es_fem & es_dmt1 & (dgis2['escolarida'] > 4) & (dgis2['escolarida'] <= 6)).astype(int)
dgis2['f_dmt1_preparatoria'] = (es_fem & es_dmt1 & (dgis2['escolarida'] > 6) & (dgis2['escolarida'] <= 8)).astype(int)
dgis2['f_dmt1_profesional'] = (es_fem & es_dmt1 & (dgis2['escolarida'] == 9)).astype(int)
dgis2['f_dmt1_posgrado'] = (es_fem & es_dmt1 & (dgis2['escolarida'] == 10)).astype(int)

dgis2['f_dmt2_sin_esc'] = (es_fem & es_dmt2 & (dgis2['escolarida'] == 1)).astype(int)
dgis2['f_dmt2_preescolar'] = (es_fem & es_dmt2 & (dgis2['escolarida'] == 2)).astype(int)
dgis2['f_dmt2_primaria'] = (es_fem & es_dmt2 & (dgis2['escolarida'] > 2) & (dgis2['escolarida'] <= 4)).astype(int)
dgis2['f_dmt2_secundaria'] = (es_fem & es_dmt2 & (dgis2['escolarida'] > 4) & (dgis2['escolarida'] <= 6)).astype(int)
dgis2['f_dmt2_preparatoria'] = (es_fem & es_dmt2 & (dgis2['escolarida'] > 6) & (dgis2['escolarida'] <= 8)).astype(int)
dgis2['f_dmt2_profesional'] = (es_fem & es_dmt2 & (dgis2['escolarida'] == 9)).astype(int)
dgis2['f_dmt2_posgrado'] = (es_fem & es_dmt2 & (dgis2['escolarida'] == 10)).astype(int)

C:\Users\super\AppData\Local\Temp\ipykernel_17240\3638821363.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dgis2['m_dmt1_sin_esc'] = (es_masc & es_dmt1 & (dgis2['escolarida'] == 1)).astype(int)
C:\Users\super\AppData\Local\Temp\ipykernel_17240\3638821363.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dgis2['m_dmt1_preescolar'] = (es_masc & es_dmt1 & (dgis2['escolarida'] == 2)).astype(int)
C:\Users\super\AppData\Local\Temp\ipykernel_17240\3638821363.py:13: SettingWithCopyWarning: 
A value is try

In [11]:
df_final2 = dgis2.groupby(['clave_municipio', 'año']).agg(
    # Conteos para Hombres DMT 1
    hombres_dmt1_sin_esc=('m_dmt1_sin_esc', 'sum'),
    hombres_dmt1_preescolar=('m_dmt1_preescolar', 'sum'),
    hombres_dmt1_primaria=('m_dmt1_primaria', 'sum'),
    hombres_dmt1_secundaria=('m_dmt1_secundaria', 'sum'),
    hombres_dmt1_preparatoria=('m_dmt1_preparatoria', 'sum'),
    hombres_dmt1_profesional=('m_dmt1_profesional', 'sum'),
    hombres_dmt1_posgrado=('m_dmt1_posgrado', 'sum'),

    hombres_dmt2_sin_esc=('m_dmt2_sin_esc', 'sum'),
    hombres_dmt2_preescolar=('m_dmt2_preescolar', 'sum'),
    hombres_dmt2_primaria=('m_dmt2_primaria', 'sum'),
    hombres_dmt2_secundaria=('m_dmt2_secundaria', 'sum'),
    hombres_dmt2_preparatoria=('m_dmt2_preparatoria', 'sum'),
    hombres_dmt2_profesional=('m_dmt2_profesional', 'sum'),
    hombres_dmt2_posgrado=('m_dmt2_posgrado', 'sum'),
    
    # Conteos para Mujeres DMT 1
    mujeres_dmt1_sin_esc=('f_dmt1_sin_esc', 'sum'),
    mujeres_dmt1_preescolar=('f_dmt1_preescolar', 'sum'),
    mujeres_dmt1_primaria=('f_dmt1_primaria', 'sum'),
    mujeres_dmt1_secundaria=('f_dmt1_secundaria', 'sum'),
    mujeres_dmt1_preparatoria=('f_dmt1_preparatoria', 'sum'),
    mujeres_dmt1_profesional=('f_dmt1_profesional', 'sum'),
    mujeres_dmt1_posgrado=('f_dmt1_posgrado', 'sum'),

    mujeres_dmt2_sin_esc=('f_dmt2_sin_esc', 'sum'),
    mujeres_dmt2_preescolar=('f_dmt2_preescolar', 'sum'),
    mujeres_dmt2_primaria=('f_dmt2_primaria', 'sum'),
    mujeres_dmt2_secundaria=('f_dmt2_secundaria', 'sum'),
    mujeres_dmt2_preparatoria=('f_dmt2_preparatoria', 'sum'),
    mujeres_dmt2_profesional=('f_dmt2_profesional', 'sum'),
    mujeres_dmt2_posgrado=('f_dmt2_posgrado', 'sum'),
    
    # Puedes seguir agregando todas las combinaciones que necesites...
).reset_index()

df_final2.columns

Index(['clave_municipio', 'año', 'hombres_dmt1_sin_esc',
       'hombres_dmt1_preescolar', 'hombres_dmt1_primaria',
       'hombres_dmt1_secundaria', 'hombres_dmt1_preparatoria',
       'hombres_dmt1_profesional', 'hombres_dmt1_posgrado',
       'hombres_dmt2_sin_esc', 'hombres_dmt2_preescolar',
       'hombres_dmt2_primaria', 'hombres_dmt2_secundaria',
       'hombres_dmt2_preparatoria', 'hombres_dmt2_profesional',
       'hombres_dmt2_posgrado', 'mujeres_dmt1_sin_esc',
       'mujeres_dmt1_preescolar', 'mujeres_dmt1_primaria',
       'mujeres_dmt1_secundaria', 'mujeres_dmt1_preparatoria',
       'mujeres_dmt1_profesional', 'mujeres_dmt1_posgrado',
       'mujeres_dmt2_sin_esc', 'mujeres_dmt2_preescolar',
       'mujeres_dmt2_primaria', 'mujeres_dmt2_secundaria',
       'mujeres_dmt2_preparatoria', 'mujeres_dmt2_profesional',
       'mujeres_dmt2_posgrado'],
      dtype='object')

In [12]:
df_final2 = df_final2[(df_final2["año"] >= 2015)]
df_final2

,clave_municipio,año,hombres_dmt1_sin_esc,hombres_dmt1_preescolar,hombres_dmt1_primaria,hombres_dmt1_secundaria,hombres_dmt1_preparatoria,hombres_dmt1_profesional,hombres_dmt1_posgrado,hombres_dmt2_sin_esc,...,mujeres_dmt1_preparatoria,mujeres_dmt1_profesional,mujeres_dmt1_posgrado,mujeres_dmt2_sin_esc,mujeres_dmt2_preescolar,mujeres_dmt2_primaria,mujeres_dmt2_secundaria,mujeres_dmt2_preparatoria,mujeres_dmt2_profesional,mujeres_dmt2_posgrado
0,2,2015,0,0,1,0,1,1,0,5,...,1,0,0,22,0,104,22,26,17,0
1,2,2016,0,0,1,3,0,1,0,6,...,0,0,0,31,0,144,27,30,17,1
2,2,2017,0,0,0,3,2,0,0,8,...,0,1,0,24,0,121,32,16,15,3
3,2,2018,0,0,3,1,0,1,0,3,...,0,0,0,20,0,128,25,1,13,1
4,2,2019,1,0,2,1,0,0,0,8,...,1,0,0,14,0,102,24,20,16,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,17,2016,0,0,0,2,0,0,0,10,...,1,0,0,29,0,151,31,22,22,2
92,17,2017,0,0,2,1,2,2,0,4,...,2,0,0,15,0,127,40,18,19,2
93,17,2018,0,0,2,1,0,1,0,6,...,0,0,0,22,1,107,26,0,20,2
94,17,2019,0,0,1,0,1,0,0,2,...,0,2,0,14,0,139,36,24,21,1


In [13]:
dgis2

,ent_resid,clave_municipio,causa_def,sexo,edad,edad_agru,dia_ocurr,mes_ocurr,año,escolarida,...,f_dmt1_preparatoria,f_dmt1_profesional,f_dmt1_posgrado,f_dmt2_sin_esc,f_dmt2_preescolar,f_dmt2_primaria,f_dmt2_secundaria,f_dmt2_preparatoria,f_dmt2_profesional,f_dmt2_posgrado
0,9,7,E117,2,68,18,21,4,2015,1,...,0,0,0,1,0,0,0,0,0,0
1,9,4,E119,2,72,19,12,11,2015,5,...,0,0,0,0,0,0,1,0,0,0
3,9,5,E116,1,87,22,22,7,2015,5,...,0,0,0,0,0,0,0,0,0,0
4,9,10,E112,1,57,16,5,7,2015,3,...,0,0,0,0,0,0,0,0,0,0
7,9,2,E117,2,68,18,19,11,2015,9,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61140,9,8,E112,1,52,15,10,8,2020,1,...,0,0,0,0,0,0,0,0,0,0
61141,9,7,E117,2,26,10,28,12,2020,6,...,0,0,0,0,0,0,1,0,0,0
61142,9,12,E117,1,50,15,16,11,2020,4,...,0,0,0,0,0,0,0,0,0,0
61144,9,17,E116,1,81,21,16,12,2020,3,...,0,0,0,0,0,0,0,0,0,0


In [14]:
#unimos los dos resumenes em uno solo

In [15]:

df_final.columns

Index(['clave_municipio', 'año', 'dmt1_total', 'dmt2_total', 'otro_total',
       'dmt1_fem', 'dmt2_fem', 'dmt1_masc', 'dmt2_masc', '0-14_sexo1_dmt1',
       '0-14_sexo1_dmt2', '0-14_sexo2_dmt1', '0-14_sexo2_dmt2',
       '15-24_sexo1_dmt1', '15-24_sexo1_dmt2', '15-24_sexo2_dmt1',
       '15-24_sexo2_dmt2', '25-59_sexo1_dmt1', '25-59_sexo1_dmt2',
       '25-59_sexo2_dmt1', '25-59_sexo2_dmt2', '60+_sexo1_dmt1',
       '60+_sexo1_dmt2', '60+_sexo2_dmt1', '60+_sexo2_dmt2', 'casos totales'],
      dtype='object')

In [16]:
df_final2.columns

Index(['clave_municipio', 'año', 'hombres_dmt1_sin_esc',
       'hombres_dmt1_preescolar', 'hombres_dmt1_primaria',
       'hombres_dmt1_secundaria', 'hombres_dmt1_preparatoria',
       'hombres_dmt1_profesional', 'hombres_dmt1_posgrado',
       'hombres_dmt2_sin_esc', 'hombres_dmt2_preescolar',
       'hombres_dmt2_primaria', 'hombres_dmt2_secundaria',
       'hombres_dmt2_preparatoria', 'hombres_dmt2_profesional',
       'hombres_dmt2_posgrado', 'mujeres_dmt1_sin_esc',
       'mujeres_dmt1_preescolar', 'mujeres_dmt1_primaria',
       'mujeres_dmt1_secundaria', 'mujeres_dmt1_preparatoria',
       'mujeres_dmt1_profesional', 'mujeres_dmt1_posgrado',
       'mujeres_dmt2_sin_esc', 'mujeres_dmt2_preescolar',
       'mujeres_dmt2_primaria', 'mujeres_dmt2_secundaria',
       'mujeres_dmt2_preparatoria', 'mujeres_dmt2_profesional',
       'mujeres_dmt2_posgrado'],
      dtype='object')

In [17]:
union = pd.merge(df_final2, df_final, on=["año", "clave_municipio"], how="outer")
union.columns

Index(['clave_municipio', 'año', 'hombres_dmt1_sin_esc',
       'hombres_dmt1_preescolar', 'hombres_dmt1_primaria',
       'hombres_dmt1_secundaria', 'hombres_dmt1_preparatoria',
       'hombres_dmt1_profesional', 'hombres_dmt1_posgrado',
       'hombres_dmt2_sin_esc', 'hombres_dmt2_preescolar',
       'hombres_dmt2_primaria', 'hombres_dmt2_secundaria',
       'hombres_dmt2_preparatoria', 'hombres_dmt2_profesional',
       'hombres_dmt2_posgrado', 'mujeres_dmt1_sin_esc',
       'mujeres_dmt1_preescolar', 'mujeres_dmt1_primaria',
       'mujeres_dmt1_secundaria', 'mujeres_dmt1_preparatoria',
       'mujeres_dmt1_profesional', 'mujeres_dmt1_posgrado',
       'mujeres_dmt2_sin_esc', 'mujeres_dmt2_preescolar',
       'mujeres_dmt2_primaria', 'mujeres_dmt2_secundaria',
       'mujeres_dmt2_preparatoria', 'mujeres_dmt2_profesional',
       'mujeres_dmt2_posgrado', 'dmt1_total', 'dmt2_total', 'otro_total',
       'dmt1_fem', 'dmt2_fem', 'dmt1_masc', 'dmt2_masc', '0-14_sexo1_dmt1',
       '0-1

In [18]:
union.to_csv("dgis_info.csv", index=False, encoding="utf-8")

In [19]:
df=dgis
valores_unicos = df['ent_resid'].unique()
print(valores_unicos)
valores_unicos = df['clave_municipio'].unique()
print(valores_unicos)
valores_unicos = df['sexo'].unique()
print(valores_unicos)
valores_unicos = df['edad'].unique()
print(valores_unicos)
valores_unicos = df['edad_agru'].unique()
print(valores_unicos)
valores_unicos = df['año'].unique()
print(valores_unicos)
valores_unicos = df['escolarida'].unique()
print(valores_unicos)

[9]
[ 7  4  5 10 14  2  6 15  3 16 12 17  9 13  8 11]
[2 1]
[ 68  72  88  87  57  63  91  77  53  54  78  59  66  69  60  49  79  81
  64  70  40  85  89  65  37  62  90  73  56  83  80  39  75  67  58  44
  96  43  98  76  82  46  61  74  86  92  50  47  95  71  51  55  36  38
  84  45  94  97  52  34  41  93 102  35  26  30  99  48  42  27  32  33
 100 101  25  31  20  23  24  21  22  29 103  18  28  14 104  17 106  10
 105 113  11   5  19  15 107  16  13]
[18 19 22 16 17 23 20 15 14 21 13 12 24 11 25 10  9  8  7 26 27  6]
[2015 2016 2017 2018 2019 2020]
[ 1  5  3  9  8  4  6 10  7  2]
